In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

### Data Prep & EDA


In [ ]:
# Load the training dataset
file_path = "Phase 1 Training Dataset.xlsx"

# Load all sheets from the Excel file
xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names

# Read all sheets into a dictionary of dataframes
nh_data = {sheet: xls.parse(sheet) for sheet in sheet_names}

In [ ]:
# Function to clean and structure each group's dataset with correctly assigned NH_IDs
def clean_group_data_fixed_nh_pattern(df, group_name):
    """
    Restructure the group dataframe into a standardized format.
    Assigns NH_ID values based on a repeating pattern every 4 columns.
    """
    # Convert column names to strings
    df.columns = df.columns.astype(str)

    # Define NH IDs based on column positions (every 4 columns = new NH)
    num_nhs = df.shape[1] // 4  # Total NHs in the group

    # Extract the data rows, skipping the first row (headers)
    df_cleaned = df.iloc[1:].reset_index(drop=True)

    # Create an empty list to store formatted data
    formatted_data = []

    # Process each NH's data
    for i in range(num_nhs):
        start_col = i * 4  # Each NH has 4 columns: Date, CNA, LPN, RN
        nh_df = df_cleaned.iloc[:, start_col:start_col + 4].copy()  # Extract relevant columns

        # Rename columns dynamically
        nh_df.columns = ["Date", "CNA", "LPN", "RN"]

        # Add NH_ID, Group, and derived features
        nh_df["NH_No"] = i + 1  # Assign NH_ID as 1, 2, 3, 4, 5 repeating
        group_numb = int(group_name.replace("Group ", ""))
        nh_df["Group"] = group_numb
            

        # Convert Date column to datetime
        nh_df["Date"] = pd.to_datetime(nh_df["Date"], errors="coerce")

        # Append formatted NH data
        formatted_data.append(nh_df)

    # Concatenate all NH data for this group
    return pd.concat(formatted_data, ignore_index=True)

# Apply corrected cleaning function to all groups and combine into a single dataset
all_data = pd.concat(
    [clean_group_data_fixed_nh_pattern(nh_data[group], group) for group in sheet_names],
    ignore_index=True
)

In [ ]:
# Convert staffing hour columns to numeric
all_data["CNA"] = pd.to_numeric(all_data["CNA"], errors="coerce")
all_data["LPN"] = pd.to_numeric(all_data["LPN"], errors="coerce")
all_data["RN"] = pd.to_numeric(all_data["RN"], errors="coerce")

In [ ]:
# Check for missing values
missing_values = all_data.isnull().sum()
print("Missing Values:\n", missing_values)

# Descriptive statistics
summary_stats = all_data.describe()
print("\nSummary Statistics:\n", summary_stats)

In [ ]:
# Plot staffing hour trends over time
plt.figure(figsize=(12, 6))
sns.lineplot(data=all_data, x="Date", y="CNA", label="CNA", alpha=0.6)
sns.lineplot(data=all_data, x="Date", y="LPN", label="LPN", alpha=0.6)
sns.lineplot(data=all_data, x="Date", y="RN", label="RN", alpha=0.6)
plt.xlabel("Date")
plt.ylabel("Staffing Hours")
plt.title("Staffing Hour Trends Over Time")
plt.legend()
plt.xticks(rotation=45)
plt.show()

# Distribution of staffing hours
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(all_data["CNA"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("CNA Staffing Hour Distribution")
sns.histplot(all_data["LPN"], bins=50, kde=True, ax=axes[1])
axes[1].set_title("LPN Staffing Hour Distribution")
sns.histplot(all_data["RN"], bins=50, kde=True, ax=axes[2])
axes[2].set_title("RN Staffing Hour Distribution")
plt.show()

# correlation Matrix
correlation_matrix_fixed = all_data[["CNA", "LPN", "RN"]].corr()

# Display correlation heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(correlation_matrix_fixed, annot=True, cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Matrix of Staffing Hours")
plt.show()


## Feature Engineering


In [ ]:
# Feature Engineering
def create_date_features(df):
    """
    Creates date-based features such as day of the week, weekend flag, and month.
    """
    df = df.copy()
    df["Day_of_Week"] = df["Date"].dt.dayofweek  # 0 = Monday, 6 = Sunday
    df["Is_Weekend"] = (df["Day_of_Week"] >= 5).astype(int)  # 1 if Saturday or Sunday
    df["Month"] = df["Date"].dt.month
    df['Week_sin'] = np.sin(2 * np.pi * df['Date'].dt.month)
    df['Week_cos'] = np.sin(2 * np.pi * df['Date'].dt.month)
    return df

def create_lag_features(df, lags=[1, 7, 14, 30]):
    """
    Creates lag features for CNA, LPN, and RN staffing hours at the group level.
    Lags are specified in the `lags` parameter.
    """
    df = df.copy()
    for lag in lags:
        for col in ["CNA", "LPN", "RN"]:
            df[f"{col}_lag_{lag}"] = df.groupby(["Group"])[col].shift(lag)
    return df


def create_rolling_features(df, window=7):
    """
    Creates rolling mean and standard deviation features for CNA, LPN, and RN.
    This captures short-term trends in staffing hours at the group level.
    """
    df = df.copy()
    for col in ["CNA", "LPN", "RN"]:
        df[f"{col}_rolling_mean_{window}"] = df.groupby(["Group"])[col].rolling(window=window).mean().reset_index(level=0, drop=True)
        df[f"{col}_rolling_std_{window}"] = df.groupby(["Group"])[col].rolling(window=window).std().reset_index(level=0, drop=True)
    return df


def apply_feature_engineering(df):
    """
    Applies all feature engineering functions to a given dataset.
    """
    df = create_date_features(df)  # Add date-based features
    df = create_lag_features(df)   # Add lag features at group level
    df = create_rolling_features(df)  # Add rolling statistics at group level
    return df


In [ ]:
# Apply feature engineering to training data at GROUP level
train_data_fe = apply_feature_engineering(all_data)

In [ ]:
# Create test dataset with correct structure: each group should have all dates
num_groups = 20
date_range = pd.date_range(start="2024-04-01", end="2024-06-30")

# Create a DataFrame with all combinations of Date and Group
test_data = pd.DataFrame([(date, group) for group in range(1, num_groups + 1) for date in date_range], 
                         columns=["Date", "Group"])

# Apply date-based features
test_data_fe = create_date_features(test_data)
test_data_fe["NH_No"] = 0

# Ensure "Group" column is a int in both datasets before merging
train_data_fe["Group"] = train_data_fe["Group"].astype(int)
test_data_fe["Group"] = test_data_fe["Group"].astype(int)

In [ ]:
# sanity check
train_data_fe[(train_data_fe["Date"] == "2024-04-01") & (train_data_fe["Group"] == 1)]

In [ ]:
# Select only the group-level columns from training (exclude NH_No)
group_level_cols = [col for col in train_data_fe.columns if col not in ["NH_No", "CNA", "LPN", "RN", "Is_Weekend", "Day_of_Week", "Month", "Week_sin", "Week_cos"]]
train_data_copy = train_data_fe[group_level_cols]

# Copy the group-level features from training data (excluding raw CNA, LPN, RN values)
group_features_train = train_data_copy.groupby(["Date", "Group"]).mean().reset_index()
# group_features_train = group_features_train.drop(columns=["NH_No"], errors="ignore")

In [ ]:
# Merge these group-level features into test data based on Group & Date
test_data_fe = test_data_fe.merge(group_features_train, on=["Group", "Date"], how="left")
test_data_fe.info()

In [ ]:
train_data_fe.info()

In [ ]:
# Output of structured data
# train_data_fe.to_csv("train_data_base2.csv", index=False)
# test_data_fe.to_csv("test_data_base2.csv", index=False)